# Intensity-Aware Conformal Prediction for Neural Point Processes

Este notebook implementa e compara três abordagens de quantificação de incerteza para Processos Pontuais Temporais (TPPs):

1.  **Baseline 1 (MC Dropout Puro):** Incerteza interna do modelo (não calibrada).
2.  **Baseline 2 (Standard Adaptive CP):** Conformal Prediction adaptativo padrão (sem informação de intensidade).
3.  **Proposta (Intensity-Aware CP):** Conformal Prediction ponderado pela intensidade $\lambda(t)$.
    *   Realizamos um **Grid Search** no parâmetro $\gamma$ (gamma) da função de peso $w = 1/\lambda^\gamma$ para encontrar o ponto ótimo de eficiência.

Dataset: `retweet` (Eventos sociais com bursts/cascatas, ideal para testar dinâmica de intensidade).

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- PATCH FP16 ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

# Injetar em RoTHP
try:
    import easy_tpp.model.torch_model.torch_rothp
    easy_tpp.model.torch_model.torch_rothp.attention = attention_fixed
except: pass

from easy_tpp.model.torch_model.torch_rothp import RoTHP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")

### 1. Configuração e Treinamento (RoTHP)
Configuramos o modelo RoTHP com Dropout e carregamos o dataset `retweet`.

In [ ]:
class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.2
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()

def collate_fn_opt(batch_list, pad_id, time_scale):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        ts = (ts - ts[0]) / time_scale
        td = td / time_scale
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

print("Loading Retweet...")
dataset = load_dataset("easytpp/retweet")
train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']

all_deltas = []
for item in train_data:
    td = item['time_since_last_event']
    all_deltas.extend([d for d in td if d > 0])
time_scale = np.mean(all_deltas)

num_types = 3
pad_id = 3
collate = lambda x: collate_fn_opt(x, pad_id, time_scale)
train_loader = DataLoader(train_data, batch_size=2048, shuffle=True, collate_fn=collate, num_workers=0)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate, num_workers=0)

def train_rothp(epochs=30):
    print(f"\n>>> Treinando RoTHP (Dropout=0.2)...")
    config = ModelConfig(num_types, pad_id)
    model = RoTHP(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        if epoch % 5 == 0:
            print(f"  Ep {epoch}: Loss {total_loss:.4f}")
    return model

model = train_rothp()

### 2. Extração de Incerteza e Intensidade
Para testar a hipótese, precisamos extrair não apenas a média ($\mu$) e desvio padrão ($\sigma$) do tempo, mas também a **intensidade estimada** ($\lambda$) no momento da predição. A intensidade será usada como peso no nosso novo score.

In [ ]:
def enable_dropout(m):
    for each_module in m.modules():
        if isinstance(each_module, nn.Dropout):
            each_module.train()

def get_predictions_with_intensity(model, loader, n_samples=20):
    model.eval()
    enable_dropout(model)
    
    all_means = []
    all_stds = []
    all_targets = []
    all_intensities = [] 
    
    print(f"Coletando estatísticas MC Dropout e Intensidade (N={n_samples})...")
    with torch.no_grad():
        for batch in tqdm(loader):
            batch = [t.to(device) for t in batch]
            pad_time, pad_delta, pad_type, mask, attn = batch
            
            # --- CORREÇÃO DE DIMENSÃO ---
            # predict_one_step_at_every_event retorna predições para seq_len-1 passos.
            # Para calcular a intensidade nesses passos, precisamos que os inputs (time, type, delta)
            # também tenham tamanho seq_len-1 (removendo o último evento, que não prevê nada).
            pad_time_in = pad_time[:, :-1]
            pad_delta_in = pad_delta[:, :-1]
            pad_type_in = pad_type[:, :-1]
            attn_in = attn[:, :-1, :-1]
            
            batch_samples = []
            lambda_samples = []
            
            for _ in range(n_samples):
                dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch)
                batch_samples.append(dtimes_pred.cpu().numpy())
                
                # Para compute_intensities, passamos os inputs cortados
                # [batch, seq-1, 1]
                dtimes_tensor = dtimes_pred.unsqueeze(-1)
                
                # [batch, seq-1, 1, num_marks]
                lambdas = model.compute_intensities_at_sample_times(
                    pad_time_in, pad_delta_in, pad_type_in, dtimes_tensor, attention_mask=attn_in
                )
                # Pegar lambda total
                # [batch, seq-1, 1]
                lambda_total = lambdas.sum(dim=-1).squeeze(-1)
                lambda_samples.append(lambda_total.cpu().numpy())

            batch_samples = np.array(batch_samples)
            lambda_samples = np.array(lambda_samples)
            
            mu = batch_samples.mean(axis=0)
            std = batch_samples.std(axis=0)
            lam = lambda_samples.mean(axis=0) 
            
            targets = pad_delta.cpu().numpy()
            mask_cpu = mask.cpu().numpy()
            
            # Alinhamento (shift de 1)
            real_targets = targets[:, 1:]
            real_mask = mask_cpu[:, 1:]
            
            bs, seq_len = real_targets.shape
            for b in range(bs):
                valid_len = int(real_mask[b].sum())
                if valid_len > 0:
                    all_means.extend(mu[b, :valid_len])
                    all_stds.extend(std[b, :valid_len])
                    all_intensities.extend(lam[b, :valid_len])
                    all_targets.extend(real_targets[b, :valid_len])
                    
    return np.array(all_means), np.array(all_stds), np.array(all_intensities), np.array(all_targets)

means, stds, lambdas, targets = get_predictions_with_intensity(model, test_loader)

### 3. Grid Search de Gamma
Aqui realizamos a busca pelo parâmetro $\gamma$ ideal na função de peso $w = 1/\lambda^\gamma$.

In [ ]:
# Configuração
ALPHA = 0.1 # Alvo: 90% Cobertura
WINDOW_SIZE = 200

# Simular Shift
split_point = len(targets) // 2
targets_shifted = targets.copy()
targets_shifted[split_point:] *= 3.0 

def evaluate(lower, upper, targets):
    covered = (targets >= lower) & (targets <= upper)
    coverage = np.mean(covered)
    width = np.mean(upper - lower)
    return coverage, width

# Grid Search
GAMMAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
results = []

print("Rodando Grid Search para encontrar o Gamma ideal...")
for gamma in GAMMAS:
    # Definir função de peso parametrizada
    # Weight = 1 / (lambda^gamma)
    weights = 1.0 / (lambdas**gamma + 1e-6)
    
    # Calcular Score Ponderado
    raw_scores = np.abs(targets_shifted - means) / (stds + 1e-6)
    weighted_scores = raw_scores * weights
    
    q_hats = []
    for t in range(len(targets_shifted)):
        if t > WINDOW_SIZE:
            recent = weighted_scores[t-WINDOW_SIZE:t]
            q_base = np.quantile(recent, 1-ALPHA)
            q_base = np.clip(q_base, 0.001, 100.0)
        else:
            q_base = 1.0
        
        # Intervalo efetivo = q_base * sigma / weight
        q_eff = q_base / (weights[t] + 1e-9)
        q_hats.append(q_eff)
    
    q_hats = np.array(q_hats)
    lower = means - q_hats * stds
    upper = means + q_hats * stds
    
    # Avaliar apenas na região de Shift (metade final)
    cov, wid = evaluate(lower[split_point:], upper[split_point:], targets_shifted[split_point:])
    results.append({'Gamma': gamma, 'Coverage': cov, 'Width': wid})
    print(f"Gamma={gamma:.1f} | Cov={cov:.1%} | Width={wid:.2f}")

# Visualizar Resultados do Grid Search
df_res = pd.DataFrame(results)
best_row = df_res.loc[df_res['Width'].idxmin()]
print(f"\nMelhor Gamma: {best_row['Gamma']} (Width={best_row['Width']:.2f})")

fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:red'
ax1.set_xlabel('Gamma (Sensibilidade à Intensidade)')
ax1.set_ylabel('Largura Média (Menor é Melhor)', color=color)
ax1.plot(df_res['Gamma'], df_res['Width'], marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Cobertura (Alvo ~90%)', color=color)
ax2.plot(df_res['Gamma'], df_res['Coverage'], marker='s', linestyle='--', color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.axhline(0.9, color='gray', linestyle=':', label='Alvo')

plt.title("Impacto do Gamma na Eficiência e Cobertura")
plt.show()

### 4. Visualização com o Melhor Modelo
Plotamos o melhor resultado encontrado comparado ao baseline (Gamma=0).

In [ ]:
# Reconstruir o melhor modelo
best_gamma = best_row['Gamma']
weights = 1.0 / (lambdas**best_gamma + 1e-6)
raw_scores = np.abs(targets_shifted - means) / (stds + 1e-6)
weighted_scores = raw_scores * weights

q_hats_best = []
for t in range(len(targets_shifted)):
    if t > WINDOW_SIZE:
        recent = weighted_scores[t-WINDOW_SIZE:t]
        q = np.quantile(recent, 1-ALPHA)
        q = np.clip(q, 0.001, 100.0)
    else:
        q = 1.0
    q_hats_best.append(q / (weights[t] + 1e-9))

q_hats_best = np.array(q_hats_best)
l_best = means - q_hats_best * stds
u_best = means + q_hats_best * stds

# Reconstruir Baseline (Gamma=0)
weights0 = 1.0 / (lambdas**0.0 + 1e-6) # = 1.0
scores0 = np.abs(targets_shifted - means) / (stds + 1e-6)
q_hats0 = []
for t in range(len(targets_shifted)):
    if t > WINDOW_SIZE:
        recent = scores0[t-WINDOW_SIZE:t]
        q = np.quantile(recent, 1-ALPHA)
        q = np.clip(q, 0.1, 10.0)
    else:
        q = 2.0
    q_hats0.append(q)
q_hats0 = np.array(q_hats0)
l0 = means - q_hats0 * stds
u0 = means + q_hats0 * stds

# Plotar
start_view = split_point + 100
end_view = start_view + 300
t_idx = np.arange(start_view, end_view)

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(t_idx, targets_shifted[start_view:end_view], 'k.', label='Real')

wid0 = df_res.loc[df_res['Gamma']==0.0, 'Width'].values[0]
ax.fill_between(t_idx, l0[start_view:end_view], u0[start_view:end_view], 
                color='blue', alpha=0.2, label=f'Standard CP (Gamma=0, W={wid0:.2f})')

ax.fill_between(t_idx, l_best[start_view:end_view], u_best[start_view:end_view], 
                color='green', alpha=0.2, label=f'Intensity CP (Gamma={best_gamma}, W={best_row["Width"]:.2f})')

ax.set_title(f"Melhoria de Eficiência com Gamma={best_gamma}")
ax.legend()
plt.show()